# GeoBench Jupyter Notebook Example

This notebook demonstrates how to use GeoBench to benchmark code execution in Jupyter notebooks.

First we define a function to benchmark:

In [ ]:
import math


def count_primes(num):
    count = 0
    for i in range(2, num):
        if all(i % j != 0 for j in range(2, int(math.sqrt(i)) + 1)):
            count += 1
    return count

## Method 1: Using the Benchmark Class

You can use the `Benchmark` class to manually start and finish benchmarking around code execution.

In [ ]:
from geobench import Benchmark

# Create a benchmark instance
benchmark = Benchmark(
    wait=2.0,               # Wait for 2 seconds at the beginning
    monitor=2.0,            # Monitor for 2 seconds before and after execution
    clear_outdir=True,      # Clean output directory if it exists
)

In [ ]:
# Start benchmarking
benchmark.start()

# Execute code to benchmark
try:
    result = count_primes(1_000_000)
    print(f"Found {result} prime numbers.")
    success = True

except Exception as err:  # noqa: BLE001
    print(f"Error: {err}")
    success = False

# Stop benchmarking
benchmark.stop()

## Method 2: Using the @benchmark decorator

For a simpler approach, you can use the `@benchmark` decorator.

Because this example uses multi-processing, `count_primes()` method defined above cannot be used directly. Instead, we import an identical function from `count_primes.py` file.

In [ ]:
from geobench import benchmark

from count_primes import count_primes


# Define a function with the benchmark decorator
@benchmark(
    wait=2.0,
    monitor=2.0,
    clear_outdir=True,
)
def parallel_count_primes(num, cores=4):
    import multiprocessing

    with multiprocessing.Pool(cores) as pool:
        results = pool.map(count_primes, [num] * cores)

    return sum(results)

In [ ]:
result = parallel_count_primes(num=1_000_000, cores=2)
print(f"Total primes counted: {result["result"]}")